In [2]:
import pandas as pd
import os
from datetime import datetime

# --- 1. RESEARCH PARAMETERS (FROM UK-DALE PAPER TABLE 4) ---
# Format: [Min Threshold (W), Max Power (W), Min Off Duration (s)]
APPLIANCE_PARAMS = {
    "Kettle":         [2000, 3100, 60],
    "Fridge":         [50,   300,  300],
    "WashingMachine": [20,   2500, 600], 
    "Microwave":      [200,  3000, 60],
    "Dishwasher":     [10,   2500, 600]
}

# Mapping appliances to their UK-DALE House 1 channels
CHANNEL_MAP = {"Kettle": 10, "Fridge": 12, "WashingMachine": 5, "Microwave": 13, "Dishwasher": 6}
HOUSE_DIR = "House_1"

# High-frequency 16kHz data true start limit (March 16, 2013)
HF_START = 1363442400 

def find_events(app_name, params):
    min_thr, max_p, min_off = params
    path = os.path.join(HOUSE_DIR, f"channel_{CHANNEL_MAP[app_name]}.dat")
    
    if not os.path.exists(path):
        print(f"⚠️ Missing channel data file: {path}")
        return []

    print(f"🔍 Hunting for {app_name} (Channel {CHANNEL_MAP[app_name]})...")
    df = pd.read_csv(path, sep=' ', header=None, names=['ts', 'p'])
    
    # Filter for timeframes where 16kHz FLACs exist
    active = df[(df['ts'] >= HF_START) & (df['p'] > min_thr)]
    
    events = []
    last_event_ts = 0
    
    for _, row in active.iterrows():
        ts = int(row['ts'])
        power = row['p']

        if (ts - last_event_ts) >= min_off and power <= max_p:
            dt = datetime.fromtimestamp(ts)
            events.append({
                "ts": ts,
                "year": dt.year,
                "week": dt.isocalendar()[1],
                "prefix": f"vi-{(ts//3600)*3600}"
            })
            last_event_ts = ts
        
        if len(events) >= 500: break # 10 train + 30 test
    return events

# --- 2. EXECUTION ---
print("🚀 Starting Clean-Slate Event Detection...")
with open("master_events_clean.txt", "w") as f:
    f.write("Appliance,Subset,Timestamp,Year,Week,File_Prefix\n")
    for app, params in APPLIANCE_PARAMS.items():
        found = find_events(app, params)
        print(f"✅ {app}: Found {len(found)} events matching criteria.")
        for i, ev in enumerate(found):
            # Enforce strict 10 train / 30 test split
            subset = "train" if i < 10 else "test"
            f.write(f"{app},{subset},{ev['ts']},{ev['year']},wk{ev['week']},{ev['prefix']}\n")

print("\n📁 Created 'master_events_clean.txt'!")

🚀 Starting Clean-Slate Event Detection...
🔍 Hunting for Kettle (Channel 10)...
✅ Kettle: Found 500 events matching criteria.
🔍 Hunting for Fridge (Channel 12)...
✅ Fridge: Found 500 events matching criteria.
🔍 Hunting for WashingMachine (Channel 5)...
✅ WashingMachine: Found 500 events matching criteria.
🔍 Hunting for Microwave (Channel 13)...
✅ Microwave: Found 500 events matching criteria.
🔍 Hunting for Dishwasher (Channel 6)...
✅ Dishwasher: Found 500 events matching criteria.

📁 Created 'master_events_clean.txt'!
